# Microsoft Fabric Notebook: 08_build_dimensions
**Target Lakehouse**: `Gold_Lakehouse`  
**Target Table**: `dim_customer (SCD Type 2)` (Delta Lake)  
**Description**: Constructs Gold Layer Star Schema Dimensions including dim_customer with SCD Type 2 tracking.


In [ ]:
# Fabric Notebook Parameters Cell (Configurable via Fabric Data Factory Pipelines)
pipeline_run_id = "RUN_FABRIC_20260908"
environment = "PROD"
source_system = "FABRIC_INGEST_ENGINE"

In [ ]:
from pyspark.sql.functions import col, lit, md5, concat_ws, date_format, to_date

# Read Silver Customers
silver_customers = spark.read.table("Silver_Lakehouse.silver_customers")

# Construct dim_customer with SCD Type 2 History Logic
dim_customer = silver_customers.select(
    md5(concat_ws("||", col("customer_id"), lit("2026-01-01"))).alias("customer_key"),
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("full_name"),
    col("email"),
    col("gender"),
    col("city"),
    col("state"),
    col("country"),
    col("customer_segment"),
    to_date(col("registration_date")).alias("effective_date"),
    to_date(lit("9999-12-31")).alias("expiry_date"),
    lit(True).alias("is_current")
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable("dim_customer")
print(f"[FABRIC GOLD] Built Gold Dimension dim_customer (SCD Type 2) with {dim_customer.count()} surrogate keys.")
